# Multi-Layer Perceptrons

In Lesson 02, we built a single Artificial Neuron. In Lesson 03, we gave it the ability to learn complex curves using Activation Functions.

But a single neuron, even with an activation function, is mathematically limited. It can only draw a single decision boundary. If you have a highly complex dataset—like recognizing a human face or translating a language—a single boundary is practically useless.

To solve complex problems, we must stack hundreds or thousands of neurons together into a **Multi-Layer Perceptron (MLP)**. This is the absolute foundation of what we call a "Deep" Neural Network.

Let's set up our PyTorch environment to build our first true Neural Network.

In [1]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

# Set professional visualization styling
sns.set_theme(style="whitegrid")

print("✅ PyTorch Multi-Layer Perceptron Environment Ready.")

✅ PyTorch Multi-Layer Perceptron Environment Ready.


# 1. The Architecture of an MLP

A Multi-Layer Perceptron (also known as a Fully Connected Network or Dense Network) is structured in distinct vertical columns called **Layers**.

1. **The Input Layer**: Not actually a layer of neurons, but simply the raw data (Features) entering the network. If your dataset has 10 columns, your input layer has 10 nodes.
2. **The Hidden Layers**: Where the mathematical "thinking" happens. They are called "hidden" because we do not directly see their outputs; they only pass data to the next layer. An MLP can have one hidden layer, or one hundred.
3. **The Output Layer**: The final set of neurons that deliver the prediction. If you are predicting House Price (Regression), it is a single neuron. If you are classifying images of Cats, Dogs, and Birds, it is 3 neurons.

**The Golden Rule of MLPs**: They are *Fully Connected*. Every single neuron in Layer $L$ connects to every single neuron in Layer $L+1$.

# 2. The Mathematics of Forward Propagation

When data moves from the Input Layer, through the Hidden Layers, and out of the Output Layer, we call this **Forward Propagation**.

Because the layers are fully connected, we do not calculate neurons one by one. We use matrix multiplication to calculate an entire layer simultaneously!

Let's define a 2-Layer Neural Network (Input $\rightarrow$ Hidden Layer 1 $\rightarrow$ Output Layer).

* Let $X$ be our input data matrix.
* Let $W^{[1]}$ and $b^{[1]}$ be the weights and biases for Hidden Layer 1.
* Let $W^{[2]}$ and $b^{[2]}$ be the weights and biases for the Output Layer.

### Step 1: Calculate Hidden Layer 1

First, we apply the Linear Transformation for the entire layer:


$$Z^{[1]} = X \cdot W^{[1]T} + b^{[1]}$$


Next, we apply the non-linear Activation Function (e.g., ReLU) to the entire layer:


$$A^{[1]} = \max(0, Z^{[1]})$$

### Step 2: Calculate the Output Layer

Now, the *activations* from Layer 1 ($A^{[1]}$) become the *inputs* for Layer 2!


$$Z^{[2]} = A^{[1]} \cdot W^{[2]T} + b^{[2]}$$


Finally, we apply the output activation (e.g., Sigmoid for a probability between 0 and 1):


$$\hat{y} = \frac{1}{1 + e^{-Z^{[2]}}}$$

*(Insight: This is why it is called a Neural "Network". The output of one mathematical equation literally becomes the input variable for the next mathematical equation, compounding in complexity.)*

# 3. Implementing Forward Propagation in PyTorch

In PyTorch, we rarely write out the raw $X \cdot W^T + b$ matrix multiplications by hand. Instead, we use the `torch.nn` module, which provides pre-built, highly optimized layers. The standard fully connected layer is called `nn.Linear`.

Let's build a Deep Neural Network to predict whether a customer will churn, based on 5 features.

* **Input Layer**: 5 features.
* **Hidden Layer 1**: 16 neurons + ReLU.
* **Hidden Layer 2**: 8 neurons + ReLU.
* **Output Layer**: 1 neuron + Sigmoid.

In [2]:
# 1. Define a Batch of Input Data
# Let's say we have 3 customers, each with 5 features
X = torch.tensor([
    [1.2, -0.5, 0.8, 2.1, -1.1], # Customer 1
    [0.0,  1.1, 0.2, 0.5,  0.9], # Customer 2
    [-1.5,-0.2, 3.1, 0.0, -0.7]  # Customer 3
], dtype=torch.float32)

print(f"Input Matrix Shape: {X.shape} (Batch Size=3, Features=5)\n")

# 2. Architect the Multi-Layer Perceptron using nn.Sequential
# Sequential automatically passes the output of one layer to the next!
model = nn.Sequential(
    # Hidden Layer 1
    # nn.Linear(in_features, out_features)
    nn.Linear(5, 16), 
    nn.ReLU(),
    
    # Hidden Layer 2
    nn.Linear(16, 8),
    nn.ReLU(),
    
    # Output Layer
    nn.Linear(8, 1),
    nn.Sigmoid() # Squashes final output to a probability (0 to 1)
)

print("--- Neural Network Architecture ---")
print(model)

# 3. Execute Forward Propagation
# We simply pass the data matrix X through the model!
predictions = model(X)

print("\n--- Final Network Predictions ---")
print("Probabilities of Churn:")
print(predictions.detach().numpy())

Input Matrix Shape: torch.Size([3, 5]) (Batch Size=3, Features=5)

--- Neural Network Architecture ---
Sequential(
  (0): Linear(in_features=5, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=8, bias=True)
  (3): ReLU()
  (4): Linear(in_features=8, out_features=1, bias=True)
  (5): Sigmoid()
)

--- Final Network Predictions ---
Probabilities of Churn:
[[0.5248314 ]
 [0.51642746]
 [0.54535705]]


# 4. Dimensionality and Matrix Shapes

The hardest part of building MLPs for beginners is getting the matrix dimensions to match. If your input has 5 features, the first `nn.Linear` layer **must** have `in_features=5`.

PyTorch handles the weight matrices ($W$) internally. When you declare `nn.Linear(5, 16)`, PyTorch automatically generates a Weight Matrix of shape `(16, 5)` and a Bias Vector of shape `(16)`.

When you pass a batch of data of shape `(3, 5)` into that layer, PyTorch executes the mathematical dot product behind the scenes:
`[3 x 5]` $\cdot$ `[5 x 16]` = `[3 x 16]`

The output shape of that layer is now exactly 16 features per customer, which flows perfectly into the next layer `nn.Linear(16, 8)`.

## Real-World Use Case or Analogy:

Think of a Multi-Layer Perceptron like a **Corporate Hierarchy processing a complex contract**:

* **The Input Layer (The Raw Data)**: A massive 500-page legal contract arrives at the company.
* **Hidden Layer 1 (The Analysts)**: 16 junior analysts read the contract. Each analyst has a different priority (Weight). Analyst A only looks for tax liabilities. Analyst B only looks for timeline clauses. They each summarize their findings into a 1-page report and pass it up.
* **Hidden Layer 2 (The Managers)**: 8 senior managers receive the 16 reports from the analysts. The managers don't read the raw 500-page contract (they never see the raw input). They look at the analysts' summaries, combine them, and form higher-level strategic opinions. They pass their 8 executive summaries up.
* **Output Layer (The CEO)**: The CEO receives the 8 executive summaries. The CEO assigns weights to the managers (e.g., they trust Manager A more than Manager B), sums up their opinions, and makes a single, final binary decision: **Sign the Contract (1)** or **Reject the Contract (0)**.

Depth (adding layers) allows the network to learn increasingly abstract and complex representations of the data!